__Prerequisites:__

* Running on Terra, as access to the TOPMed genotype data requires the `tnu` command.
* Access to the [`mgb-KEW-K01-GCP/gxpa-lcms-mediation`](https://app.terra.bio/#workspaces/mgb-KEW-K01-GCP/gxpa-lcms-mediation) workspace where metadata files are stored.

In [ ]:
library(data.table)
if(!file.exists("bcftools/bcftools")) system("git clone --recurse-submodules https://github.com/samtools/htslib.git; git clone https://github.com/samtools/bcftools.git; cd bcftools; make")

vs  <- fread(cmd='gcloud storage cat gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/variants_of_interest.csv')
drs <- fread(cmd='gcloud storage cat gs://fc-secure-4a392455-5587-4d6f-b8bd-01a1f834ae63/geno_files_drs.csv')
dt  <- drs[vs,on='chr', allow.cartesian=T]

# Extract dosages

In [ ]:
get_dosage <- \(vcf_url, idx_fnm, chr) {
  fread(cmd = paste0(
    "bcftools/bcftools view",
      " -R ",chr,"_var_regions.txt",
      " -i'ID=@",chr,"_var_ids.txt'",
      " '",vcf_url,"##idx##",idx_fnm,"'", # vcf_url could have funny characters, so wrap in single quotes so the shell won't interpret them.
    " | ",
    "bcftools/bcftools +bcftools/plugins/dosage.so"
))}

dosages <- dt[, by=chr, {
  writeLines(paste0(chr,'\t',pos), paste0(chr[1],'_var_regions.txt'))
  writeLines(rsid,                 paste0(chr[1],'_var_ids.txt'    ))

  # Must run single-threaded or else bcftools has read errors.
  Map(unique(vcf_drs), unique(csi_drs), f=\(vcf_drs,csi_drs) {
    vcf_url <- system(paste('tnu drs access', vcf_drs), intern=T)
    csi_url <- system(paste('tnu drs access', csi_drs), intern=T)
    system(paste0("curl '",csi_url,"' -o idx.csi")) # Download VCF index files for fast random access.
    dos <- get_dosage(vcf_url,   'idx.csi', chr[1])
    dos[, names(.SD) := lapply(.SD,as.character)] # CHROM,POS,REF,ALT to character. Because if no variants are found they will be of type logical, causing merge errors.
  }) |> Reduce(f=\(x,y) merge(x,y,all=T))
}]

# Format

In [ ]:
dosages2 <- dosages[
  ][vs[,pos:=as.character(pos)], on=c(`#[1]CHROM`='chr', `[2]POS`='pos')
  ][, `:=`(chr=`#[1]CHROM`, pos=`[2]POS`)
  ][, names(.SD) := lapply(.SD,as.numeric), .SDcols=patterns('NWD')
  ][, cpaid := paste(chr, pos, ref, alt, sep='_')

  # If alt is blank, allow whatever alelle. But if not, make sure alt/ref match (allowing flips).
  ][alt=='' | ((ref==`[3]REF` & alt==`[4]ALT`) |
               (ref==`[4]ALT` & alt==`[3]REF`))

  ][!duplicated(cpaid)
  ][, .SD, .SDcols=patterns('cpaid|NWD')
] |> transpose(make.names='cpaid', keep.names='NWD_Id')
dosages2[, NWD_Id := sub('^\\[[0-9]+\\]', '', NWD_Id) ] # "[123]NWD987" -> "NWD987"
dosages2[, names(dosages2)[sapply(dosages2, \(col) all(is.na(col)))] := NULL] # Rm all-NA SNPs.

In [ ]:
head(dosages2)

# Write

In [ ]:
dir.create('data/raw/genomics',rec=T)
fwrite(dosages2, 'data/raw/genomics/dosages.csv')
system('gcloud storage cp data/raw/genomics/dosages.csv $WORKSPACE_BUCKET/data/raw/genomics/dosages.csv')